# TinyThinker · GRPO 后训练（Kaggle 免费版）

**开跑前必须确认的三件事**（漏一个都会白烧配额）：

1. 右上角 Settings → **Accelerator 选 GPU T4 ×2**（本项目只用单卡，选它即可）
2. Settings → **Internet 设为 On**，否则 pip / HuggingFace 下载会失败
3. 账号必须完成**手机号验证**，否则加速器选项是灰的

**配额守财三条：**
- 单次会话 ≤ 12 小时，本 notebook 默认预算 **10.5 小时**，到点自动优雅存盘
- 周配额约 30 GPU 小时，**不开着 notebook 发呆**——闲置也扣配额
- 跑完务必把 `PTL_INTERRUPT_DIR` 里的 ckpt 推到 HF 私有仓，会话结束就丢了

In [ ]:
# Step 0 —— 环境探测（先跑这个，确认 GPU 到账再往下走）
!nvidia-smi
!python -c "import torch;print(torch.__version__, torch.cuda.is_available())"

In [ ]:
# Step 1 —— 拉取代码并安装依赖（约 3-5 分钟）
# 推荐先把 posttrain-lab 推到你自己的 GitHub，再改这里的地址
import os
!git clone https://github.com/xyx1333355555/posttrain-lab.git 2>/dev/null || echo '仓库不存在则手动上传'
%cd posttrain-lab
!pip install -q -r requirements.txt
!pip install -q bitsandbytes accelerate peft trl vllm

In [ ]:
# Step 2 —— 自动识别环境，看看系统给你配了什么方案
import sys
sys.path.insert(0, 'src')
from posttrain_lab.runtime import summary
info = summary()
for k, v in info.items():
    print(f'{k:20s} {v}')

In [ ]:
# Step 3 —— 最小闭环 smoke test（强烈建议第一次必跑，约 15 分钟）
# 目的：确认 T4 上的 fp16 + sdpa 组合能跑通，再谈正式训练
!python scripts/00_smoke_test.py --model Qwen/Qwen2.5-0.5B-Instruct --steps 20

In [ ]:
# Step 4 —— 正式 GRPO（默认 10.5h 预算，到点自动存盘）
# 16GB 显存 → 自动走 configs/grpo_16gb.yaml：0.5B + fp16 + sdpa
!python scripts/99_run_free_tier.py --stage grpo --hours 10.5

In [ ]:
# Step 5 —— 评测，拿到简历上要写的那组数字
!python scripts/04_eval.py --ckpt outputs/grpo_16gb/main/final --tasks gsm8k --limit 300 --out outputs/eval
import json, glob
for f in glob.glob('outputs/eval/*.json'):
    print(f)
    print(json.dumps(json.load(open(f)), ensure_ascii=False, indent=2))

In [ ]:
# Step 6 —— 别让结果随会话消失：推到 HF 私有仓
# 先在 Kaggle 的 Add-ons → Secrets 里添加 HF_TOKEN
from kaggle_secrets import UserSecretsClient
import os
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

!python -c "from posttrain_lab.runtime import save_checkpoint_hub; save_checkpoint_hub('outputs/grpo_16gb', 'xyx/tinythinker-ckpt')"

## 下周继续跑（断点续跑）

```
!python scripts/99_run_free_tier.py --stage grpo --hours 10.5 --resume outputs/grpo_16gb/interrupt
```

## 配额用完了怎么办

按优先级切换到其他免费额度：
1. **魔搭 ModelScope**（A10 24GB，新用户 36 小时，国内 CDN 快，可跑 1.5B 主实验）
2. **Google Colab**（T4，周配额动态）
3. **腾讯 Cloud Studio**（每月 10000 分钟 T4）
4. **阿里天池实验室**（60 小时 V100/P100/T4）

⚠️ 不要去百度飞桨 AI Studio：它限 PaddlePaddle 框架，跑 PyTorch 会被限制或 kill。